<a href="https://colab.research.google.com/github/fino15/Machine-Learning-Assignment/blob/main/Machine_learning_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('/content/drive/MyDrive/Colab/Project 1 - Weather Dataset.csv.csv')
# df = pd.read_csv('/content/Project 1 - Weather Dataset.csv.csv')
print(" Dataset successfully loaded into 'df'!\n")

 Dataset successfully loaded into 'df'!



In [14]:
#Step 1: Data cleaning
print("Checking for missing values")
#Count missing values in each column
missing_values = df.isnull().sum()
print(missing_values)

Checking for missing values
Date/Time           0
Temp_C              0
Dew Point Temp_C    0
Rel Hum_%           0
Wind Speed_km/h     0
Visibility_km       0
Press_kPa           0
Weather             0
dtype: int64


In [15]:
print("\n2.Checking for deplicate rowss")
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")


2.Checking for deplicate rowss
Number of duplicate rows: 0


In [16]:
#Drop duplicates if any exist
if duplicate_count > 0:
  df.drop_dyplicates(inplace=True)
  print("Duplicate rows dropped successfully!")
else:
  print("No duplicate rows found!")

print("\n3.Verifying structural Dta Types")
print(df.dtypes)

No duplicate rows found!

3.Verifying structural Dta Types
Date/Time            object
Temp_C              float64
Dew Point Temp_C    float64
Rel Hum_%             int64
Wind Speed_km/h       int64
Visibility_km       float64
Press_kPa           float64
Weather              object
dtype: object


In [17]:
df['Weather'].value_counts()

,count
Weather,
Mainly Clear,2106
Mostly Cloudy,2069
Cloudy,1728
Clear,1326
Snow,390
Rain,306
Rain Showers,188
Fog,150
"Rain,Fog",116


In [7]:
#Data Transformation

from sklearn.preprocessing import LabelEncoder

print("Performing Feature Engineering on Date/Time")
df['Date/Time']= pd.to_datetime(df['Date/Time'])


df['Month'] = df['Date/Time'].dt.month
df['Hour'] = df['Date/Time'].dt.hour

#Drop the original text date column
df.drop('Date/Time', axis=1, inplace=True)
print("Extracted 'Month' and 'Hour' columns")

print("\nEncoding Text Labels in the Weather Column")
label_encoder = LabelEncoder()
df['Weather_Encoded'] = label_encoder.fit_transform(df['Weather'])

#Preview of newly transformed dataset
print("\nTransformed Dataset:")
print(df[['Month', 'Hour', 'Temp_C', 'Weather','Weather_Encoded']].head())


Performing Feature Engineering on Date/Time
Extracted 'Month' and 'Hour' columns

Encoding Text Labels in the Weather Column

Transformed Dataset:
   Month  Hour  Temp_C               Weather  Weather_Encoded
0      1     0    -1.8                   Fog                7
1      1     1    -1.8                   Fog                7
2      1     2    -1.8  Freezing Drizzle,Fog                9
3      1     3    -1.5  Freezing Drizzle,Fog                9
4      1     4    -1.5                   Fog                7


In [8]:
#Step 3: Scaling and Data Reduction

from sklearn.preprocessing import StandardScaler
print("Data Reduction")
if 'Weather' in df.columns:
  df.drop('Weather', axis=1 , inplace=True)
  print("Dropped original string 'Weather' column")

  print("\nFeature Scaling")
  features_to_scale = ['Temp_C', 'Dew Point Temp_C','Rel Hum_%','Wind Speed_km/h','Press_kPa','Visibility_km','Month','Hour']

scaler =StandardScaler()
df[features_to_scale] = scaler.fit_transform(df[features_to_scale])

print("Final Ppipeline Preview")
print(df.head())

Data Reduction
Dropped original string 'Weather' column

Feature Scaling
Final Ppipeline Preview
     Temp_C  Dew Point Temp_C  Rel Hum_%  Wind Speed_km/h  Visibility_km  \
0 -0.906815         -0.593184   1.097553        -1.259808      -1.557954   
1 -0.906815         -0.574805   1.156662        -1.259808      -1.557954   
2 -0.906815         -0.547238   1.274879        -0.914513      -1.874862   
3 -0.881146         -0.528860   1.215770        -1.029611      -1.874862   
4 -0.881146         -0.538049   1.215770        -0.914513      -1.811480   

   Press_kPa     Month      Hour  Weather_Encoded  
0   0.223206 -1.597591 -1.661325                7  
1   0.223206 -1.597591 -1.516862                7  
2   0.246904 -1.597591 -1.372399                9  
3   0.258753 -1.597591 -1.227936                9  
4   0.211358 -1.597591 -1.083473                7  


# Part 2 : Training model


In [9]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import KFold, cross_val_score

In [10]:
x = df.drop(columns=['Weather_Encoded'])
y = df['Weather_Encoded']

In [11]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {}
models['knn'] = KNeighborsClassifier()
models['lgr'] = LogisticRegression(max_iter=1000)
models['gnb'] = GaussianNB()
models['dtc'] = DecisionTreeClassifier()
models['svc'] = SVC()
models['rfc'] = RandomForestClassifier()
models['gbc'] = GradientBoostingClassifier()
models['mlp'] = MLPClassifier(max_iter=500)

for n in models:
        scores = cross_val_score(models[n], x, y, cv=kf, n_jobs=-1)
        print(f'{n}: {scores.mean():.3%} +/- {scores.std():.3%}')

knn: 49.476% +/- 0.861%
lgr: 38.229% +/- 0.901%
gnb: 33.629% +/- 0.649%
dtc: 47.006% +/- 0.899%
svc: 43.386% +/- 1.325%
rfc: 57.866% +/- 1.113%
gbc: 38.331% +/- 10.632%
mlp: 44.695% +/- 0.854%
